<a href="https://colab.research.google.com/github/kaykesm/MLCB_CK_2026/blob/main/AULA_07/lab03_aula07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =====================================================================
# LAB 3: CHATBOT DE E-COMMERCE COM DECISION TREE
# =====================================================================

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier

dados_treino = [
    ("Onde esta meu pedido e qual o rastreio", "rastrear_pedido"),
    ("Quero saber a localizacao da minha encomenda", "rastrear_pedido"),
    ("Quero cancelar minha compra e pedir reembolso", "cancelar_compra"),
    ("Preciso cancelar o pedido feito ontem", "cancelar_compra"),
    ("Qual o horario de atendimento da loja fisica", "fora_escopo")
]

stopwords = ["o", "a", "meu", "minha", "e", "da", "do"]

# 1. PRÉ-PROCESSAMENTO
def pré_processar(texto):
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower()).strip()
    tokens = texto_limpo.split()

    # TODO 3.1
    tokens_filtrados = [token for token in tokens if token not in stopwords]

    return " ".join(tokens_filtrados)

X_treino_limpo = [pré_processar(item[0]) for item in dados_treino]
y_treino = [item[1] for item in dados_treino]

# 2. VETORIZAÇÃO E TREINAMENTO
vectorizer = TfidfVectorizer()
X_vetorizado = vectorizer.fit_transform(X_treino_limpo)

# TODO 3.2
modelo_tree = DecisionTreeClassifier()

# TODO 3.3
modelo_tree.fit(X_vetorizado, y_treino)

# 3. EXTRAÇÃO DE ENTIDADE
def extrair_codigo_rastreio(texto):
    match = re.search(r'BR\d{9}', texto, re.IGNORECASE)

    # TODO 3.4
    if match:
        return match.group(0).upper()
    else:
        return None

# 4. PIPELINE NLU
def processar_ecommerce(mensagem_usuario, threshold=0.50):
    texto_p = pré_processar(mensagem_usuario)
    vetor_input = vectorizer.transform([texto_p])

    probas = modelo_tree.predict_proba(vetor_input)[0]

    maior_confianca = max(probas)
    intencao_prevista = modelo_tree.classes_[probas.argmax()]

    # TODO 3.5
    if maior_confianca < threshold or intencao_prevista == "fora_escopo":
        return {
            "status": "FALLBACK"
        }
    else:
        return {
            "status": "SUCESSO",
            "intencao": intencao_prevista,
            "confianca": maior_confianca,
            "codigo_rastreio": extrair_codigo_rastreio(mensagem_usuario)
        }

# --- TESTE 3 ---
if __name__ == "__main__":
    print(processar_ecommerce("Quero rastrear o pedido BR987654321"))

{'status': 'SUCESSO', 'intencao': np.str_('rastrear_pedido'), 'confianca': np.float64(1.0), 'codigo_rastreio': 'BR987654321'}


In [2]:
print(extrair_codigo_rastreio("Quero rastrear o pedido BR987654321"))

BR987654321
